# 01 Used Car Price Prediction: Initial EDA and Cleaning #

**Project question:** Can we estimate a fair used-car listing price from vehicle characteristics such as make, model, year, mileage, fuel type, transmission, drivetrain, and location?

**Target variable:** `price`

**Notebook output:** `../data/processed.csv`

**Important rule:** I do not overwrite `raw.csv`. The raw dataset stays unchanged so the workflow remains reproducible.

In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

Loading the CSV, fixing d-type error

In [7]:
df = pd.read_csv('../data/raw.csv', 
        dtype={
            'fuel_type': 'string',
            'engine_block': 'string'
}, 
low_memory=False)

In [8]:
df.dtypes

id                  str
vin                 str
price           float64
miles           float64
stock_no            str
year            float64
make                str
model               str
trim                str
body_type           str
vehicle_type        str
drivetrain          str
transmission        str
fuel_type        string
engine_size     float64
engine_block     string
seller_name         str
street              str
city                str
state               str
zip                 str
dtype: object

Inspection, answering what Columns exists, which are useful/useless, or are there impossible values etc.

In [9]:
df.shape

(393603, 21)

In [ ]:
df.info()

In [39]:
df.describe()

,price,miles,year,engine_size
count,3.584860e+05,3.665900e+05,393586.000000,320950.000000
mean,2.601902e+04,7.566339e+04,2016.414829,2.785073
std,2.064007e+04,5.775442e+04,3.345400,1.236639
min,0.000000e+00,0.000000e+00,1981.000000,0.600000
25%,1.490000e+04,3.491375e+04,2015.000000,2.000000
50%,2.190000e+04,6.232800e+04,2017.000000,2.400000
75%,3.199500e+04,1.032830e+05,2019.000000,3.500000
max,1.288888e+06,2.300033e+06,2022.000000,8.400000


In [ ]:
df.isna().sum().sort_values(ascending=False)

In [15]:
duplicate_count = df.duplicated().sum()
duplicate_count

np.int64(0)

### Initial observations

Write this in your own notebook after looking at the outputs above.

Example wording:

- The dataset contains 393603 rows and 21 columns.
- The target column is `price`.
- Several columns have missing values, especially `engine_block`, `engine_size`, and `fuel_type`.
- From .describe() we notice that there are no impossible years, impossible mileage, or impossible engine sizes. 
- The first cleaning priorities are the null values, D-type errors in engine_block and fuel type, and impossible prices.

## Target Variable check: `price`

In [ ]:
TARGET = 'price'
df[TARGET].describe()

In [30]:
df[TARGET].isna().sum()

np.int64(27013)

In [ ]:
#Checking for negative or zero values in the target variable
df.loc[(df[TARGET] <= 0), [TARGET]].head(20)


### There are 11 values with a value of 0.0

I will remove rows where `price` is missing or less than or equal to zero because these rows cannot be used to train a supervised price-prediction model. I will not automatically remove expensive vehicles only because they are expensive. High prices may represent real luxury or collectible vehicles, so I will inspect extreme values before deciding whether to cap or remove them.

## Check Categorical Columns
Standardize Text-columns before modelling
Need to check common col values to see that there are no `_  Honda ` or `Honda  ` treated differently from `Honda` else we strip.

In [ ]:
text_cols = df.select_dtypes(include=["object", "string"]).columns.tolist()
text_cols

In [ ]:
for col in text_cols[:15]:
    print(f"\n--- {col} ---")
    display(df[col].value_counts(dropna=False).head(15))

### Categorical cleaning decision

I will standardize text columns by converting values to lowercase and stripping extra spaces. Rare makes or models will be handled during feature engineering. My goal is to retain as much information as possible in the dataset.
